In [8]:
"""
Day 2 - Sales Analytics Without Pandas
Squad Python Specialization - Data Science, AI & Analytics

Tasks 1-7 use only the standard library. pandas is allowed in task 8, to check your work.
"""
import csv
from pathlib import Path

DATA_PATH = Path("data") / "electronics_sales.csv"


def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    # """Read the CSV with the csv module and return the raw rows."""
    with open(path, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

    print(f"Number of rows found: {len(rows)}")
    print(f"Column names found: {reader.fieldnames}")

    return rows
    # raise NotImplementedError
read_sales(DATA_PATH)


Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


[{'sale_id': 'S-3001',
  'sale_date': '2026-09-01',
  'store': 'Westgate',
  'product': 'Safaricom Neon Ray',
  'quantity': '1',
  'unit_price_kes': '5999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3002',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'USB-C Charger 25W',
  'quantity': '2',
  'unit_price_kes': '1499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3003',
  'sale_date': '2026-09-02',
  'store': 'Kisumu Mega Plaza',
  'product': 'Lenovo Tab M10',
  'quantity': '2',
  'unit_price_kes': '24999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3004',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'Xiaomi Redmi Buds',
  'quantity': '2',
  'unit_price_kes': '3199',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3005',
  'sale_date': '2026-09-03',
  'store': 'Nakuru Westside',
  'product': 'Tecno Spark 20',
  'quantity': '2',
  'unit_price_kes': '13499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3006',
  'sale_date': '2026-09-03',
  's

In [10]:
def parse_number(value: str | None) -> float | None:
    """Convert a messy numeric string to a float, or return None if invalid."""
    if value is None:
        return None

    cleaned = value.strip().upper().replace("KES", "").strip()
    cleaned = cleaned.replace(",", "")

    word_to_num = {
        "ZERO": 0,
        "ONE": 1,
        "TWO": 2,
        "THREE": 3,
        "FOUR": 4,
        "FIVE": 5,
        "SIX": 6,
        "SEVEN": 7,
        "EIGHT": 8,
        "NINE": 9,
        "TEN": 10
    }

    if cleaned in ("", "N/A", "NONE", "NULL", "-"):
        return None

    if cleaned in word_to_num:
        return word_to_num[cleaned]


    try:
        return float(cleaned)
    except ValueError:
        return None


print(parse_number("two"))  # Returns: 2.0
print(parse_number("  KES 1,200.50  "))  # Returns: 1200.50
print(parse_number("    N/A"))  # Returns: None

    
    # """Convert a messy numeric string to a float, or return None if it cannot be trusted."""
    # raise NotImplementedError

2
1200.5
None


In [16]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    """Return clean rows and skipped rows with reasons."""
    clean_rows = []
    skipped_rows = []

    def find_value(row: dict[str, str], *names: str) -> str | None:
        normalized = {
            key.lower().replace("_", "").replace(" ", ""): value
            for key, value in row.items()
        }

        for name in names:
            target = name.lower().replace("_", "").replace(" ", "")
            # FIX: Match exact string OR fallback to a substring lookup (e.g., 'unitprice' matches 'unitpricekes')
            if target in normalized:
                return normalized[target]
            for key, value in normalized.items():
                if target in key:
                    return value
        return None

    for row_number, row in enumerate(rows, start=1):
        quantity_raw = find_value(row, "quantity", "qty")
        unit_price_raw = find_value(row, "unit_price", "unit price", "price")
        revenue_raw = find_value(row, "revenue", "total sales", "sales")

        quantity = parse_number(quantity_raw)
        unit_price = parse_number(unit_price_raw)
        revenue = parse_number(revenue_raw)

        if quantity is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid quantity", "data": row}
            )
            continue

        if unit_price is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid unit price", "data": row}
            )
            continue

        if revenue is None and (
            revenue_raw is None or not str(revenue_raw).strip()
        ):
            revenue = quantity * unit_price
        elif revenue is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid revenue", "data": row}
            )
            continue

        clean_row = dict(row)
        clean_row["quantity"] = quantity
        clean_row["unit_price"] = unit_price
        clean_row["revenue"] = revenue
        clean_rows.append(clean_row)

    return clean_rows, skipped_rows

In [17]:
def total_revenue(rows: list[dict]) -> float:
    """Calculate total revenue as quantity × unit price."""
    total = 0.0

    for row in rows:
        quantity = row.get("quantity")
        unit_price = row.get("unit_price", row.get("unit_price_ke"))

        if not isinstance(quantity, (int, float)):
            quantity = parse_number(str(quantity))
        if not isinstance(unit_price, (int, float)):
            unit_price = parse_number(str(unit_price))

        if quantity is not None and unit_price is not None:
            total += quantity * unit_price

    return total


print(f"Total Revenue: KES {total_revenue(convert_rows(read_sales())[0]):,.2f}")

Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
Total Revenue: KES 1,317,210.00


In [20]:
clean_rows, skipped_rows = convert_rows(read_sales(DATA_PATH))

def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    """Calculate total revenue grouped by a field."""
    totals = {}

    for row in rows:
        key = row.get(field)
        if key is None:
            continue

        totals[key] = totals.get(key, 0.0) + float(row.get("revenue", 0.0))

    return totals

# Testing and calling the function
product_totals = revenue_by(clean_rows, "product")
store_totals = revenue_by(clean_rows, "store")
payment_totals = revenue_by(clean_rows, "payment_method")

print("Product revenue breakdown:")
for product, revenue in sorted(
    product_totals.items(), key=lambda item: item[1], reverse=True
):
    print(f"{product}  KES {revenue:,.2f}")

Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
Product revenue breakdown:
HP 15 Laptop  KES 389,994.00
Hisense 43in Smart TV  KES 272,993.00
Lenovo Tab M10  KES 199,992.00
Tecno Spark 20  KES 161,988.00
Infinix Hot 40  KES 63,996.00
Samsung Galaxy A15  KES 56,997.00
JBL Go 3 Speaker  KES 49,491.00
Anker Power Bank 20000  KES 38,691.00
Xiaomi Redmi Buds  KES 31,990.00
Oraimo FreePods  KES 19,593.00
USB-C Charger 25W  KES 19,487.00
Safaricom Neon Ray  KES 11,998.00


In [23]:
def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    """Return the n highest (key, value) pairs, highest first."""
    # Multi-level sort:
    # 1. Primary: Sort by revenue descending (negative float values for reverse sorting)
    # 2. Secondary: Sort by product name alphabetically (A-Z) to break ties cleanly
    sorted_items = sorted(
        totals.items(), key=lambda item: (-item[1], item[0].lower())
    )

    return sorted_items[:n]


# --- Verification with Product Dataset ---
# Assuming product_totals was calculated from the previous step
top_three = top_n(product_totals, n=3)

print("Top 3 Highest Revenue Products")
for rank, (product, revenue) in enumerate(top_three, start=1):
    print(f"{rank}. {product} — KES {revenue:,.2f}")

Top 3 Highest Revenue Products
1. HP 15 Laptop — KES 389,994.00
2. Hisense 43in Smart TV — KES 272,993.00
3. Lenovo Tab M10 — KES 199,992.00


In [26]:

# 1. List comprehension to filter high-value transactions
def high_value(rows, threshold):
    return [row for row in rows if row["amount"] > threshold]

# Define threshold
threshold = 50000
qualifying_sales = high_value(rows, threshold)

# 2. Dict comprehension to count qualifying sales per store
unique_stores = {row["store"] for row in qualifying_sales}
store_counts = {store: sum(1 for row in qualifying_sales if row["store"] == store) for store in unique_stores}

# Print qualifying sales
print(f"--- Qualifying Sales (Threshold > KES {threshold:,}) ---")
for sale in qualifying_sales:
    print(f"Store: {sale['store']} | Item: {sale['item']} | Amount: KES {sale['amount']:,}")

print("\n--- High-Value Sales Count per Store ---")
for store, count in store_counts.items():
    print(f"{store}: {count} sale(s)")

--- Qualifying Sales (Threshold > KES 50,000) ---
Store: Nairobi CBD | Item: Laptop | Amount: KES 75,000
Store: Mombasa | Item: Smart TV | Amount: KES 60,000
Store: Westlands | Item: Smartphone | Amount: KES 85,000
Store: Kisumu | Item: Refrigerator | Amount: KES 95,000
Store: Nairobi CBD | Item: Office Chair | Amount: KES 52,000

--- High-Value Sales Count per Store ---
Kisumu: 1 sale(s)
Nairobi CBD: 2 sale(s)
Westlands: 1 sale(s)
Mombasa: 1 sale(s)


In [30]:
# Reusable statistics - implement these yourself, no statistics module.
def mean(values: list[float]) -> float:
    """Calculate the arithmetic mean of a list of numbers."""
    if not values:
        raise ValueError("The list of values is empty.")
    return sum(values) / len(values)



def median(values: list[float]) -> float:
    """Calculate the median value of a list of numbers."""
    if not values:
        raise ValueError("The list of values is empty.")

    sorted_values = sorted(values)
    n = len(sorted_values)
    mid_index = n // 2

    if n % 2 == 0:  # Even number of elements
        return (sorted_values[mid_index - 1] + sorted_values[mid_index]) / 2
    else:  # Odd number of elements
        return sorted_values[mid_index]

def minimum_maximum(values: list[float]) -> tuple[float, float]:
    """Find the minimum and maximum values without using min() or max() built-ins."""

    if not values:
        raise ValueError("The list of values is empty.")

    min_value = float('inf')
    max_value = float('-inf')

    for value in values:
        if value < min_value:
            min_value = value
        if value > max_value:
            max_value = value

    return min_value, max_value

# Assuming 'clean_rows' comes from your salvaged convert_rows(read_sales()) pipeline
# Extract revenue array using a list comprehension
revenue_stream = [row["revenue"] for row in clean_rows]

# Compute metrics
revenue_mean = mean(revenue_stream)
revenue_median = median(revenue_stream)
revenue_min, revenue_max = minimum_maximum(revenue_stream)

# Execution Printout
print("--- PER-TRANSACTION REVENUE STATISTICS ---")
print(f"Mean Transaction Value   : KES {revenue_mean:,.2f}")
print(f"Median Transaction Value : KES {revenue_median:,.2f}")
print(f"Minimum Transaction Value: KES {revenue_min:,.2f}")
print(f"Maximum Transaction Value: KES {revenue_max:,.2f}")

--- PER-TRANSACTION REVENUE STATISTICS ---
Mean Transaction Value   : KES 28,025.74
Median Transaction Value : KES 10,998.00
Minimum Transaction Value: KES 1,499.00
Maximum Transaction Value: KES 259,996.00


In [32]:
import pandas as pd
import numpy as np
from pathlib import Path

def verify_with_pandas(path: Path, my_results: dict) -> None:
    """Recompute the headline numbers with pandas and print whether each one matches."""
    df = pd.read_csv(path)

    word_to_num = {
        "ONE": 1.0,
        "TWO": 2.0,
        "THREE": 3.0,
        "FOUR": 4.0,
        "FIVE": 5.0,
        "SIX": 6.0,
        "SEVEN": 7.0,
        "EIGHT": 8.0,
        "NINE": 9.0,
        "TEN": 10.0,
    }

    def clean_quantity(val):
        if pd.isna(val):
            return np.nan
        cleaned = str(val).strip().upper()
        if cleaned in word_to_num:
            return word_to_num[cleaned]
        try:
            return float(cleaned)
        except ValueError:
            return np.nan

    def clean_unit_price(val):
        if pd.isna(val):
            return np.nan
        cleaned = str(val).strip().upper().replace("KES", "").replace(",", "")
        if cleaned in ("", "N/A", "NONE", "NULL", "-", "nan"):
            return np.nan
        try:
            return float(cleaned)
        except ValueError:
            return np.nan

    df["qty_clean"] = df["quantity"].apply(clean_quantity)

    price_col = [col for col in df.columns if "price" in col.lower()][0]
    df["price_clean"] = df[price_col].apply(clean_unit_price)

    df_clean = df.dropna(subset=["qty_clean", "price_clean"]).copy()
    df_clean["revenue"] = df_clean["qty_clean"] * df_clean["price_clean"]

    pandas_results = {
        "total_revenue": float(df_clean["revenue"].sum()),
        "mean": float(df_clean["revenue"].mean()),
        "median": float(df_clean["revenue"].median()),
        "min": float(df_clean["revenue"].min()),
        "max": float(df_clean["revenue"].max()),
    }

    print("--- PANDAS AUTOMATED AUDIT METRICS ---")
    for metric, pd_val in pandas_results.items():
        my_val = my_results.get(metric)

        if my_val is None:
            status = "MISMATCH (Metric missing from custom results)"
        elif np.isclose(pd_val, my_val, atol=0.01):
            status = "MATCH"
        else:
            status = f"MISMATCH (Pandas: {pd_val:,.2f} vs Custom: {my_val:,.2f})"

        print(f"{metric.upper().replace('_', ' '):<15} : {status}")

verify_with_pandas(
    DATA_PATH,
    {
        "total_revenue": total_revenue(clean_rows),
        "mean": revenue_mean,
        "median": revenue_median,
        "min": revenue_min,
        "max": revenue_max,
    },
)


--- PANDAS AUTOMATED AUDIT METRICS ---
TOTAL REVENUE   : MATCH
MEAN            : MATCH
MEDIAN          : MATCH
MIN             : MATCH
MAX             : MATCH
